# Chapitre 10 - Exercices 10.7 à 10.12

Ce notebook traite les exercices 10.7 à 10.12 : valeurs des feuilles pour la perte
exponentielle, pas de Newton pour la déviance multinomiale, dérivation de l'algorithme 10.4
(gradient boosting à $K$ classes), le cas $K = 2$, le calcul rapide de la dépendance
partielle pour les arbres, et un exemple gaussien où l'espérance conditionnelle « invente »
un effet.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

from esl import datasets
from esl.ch10_boosting import (AdaBoostM1, GradientBoosting, GradientBoostingMulticlass,
                               RegressionTree, simulate_nested_spheres,
                               partial_dependence_bruteforce)

## Exercice 10.7 : Valeurs des feuilles pour la perte exponentielle

**Énoncé (reformulé).** Établir l'expression (10.32) : lorsque les régions $R_{jm}$ d'un arbre
sont fixées, la solution de (10.30) pour la perte exponentielle est le log-rapport de cotes
pondéré dans chaque région,

$$\hat\gamma_{jm} = \frac12\log\frac{\sum_{x_i\in R_{jm}} w_i^{(m)} I(y_i = 1)}{\sum_{x_i\in R_{jm}} w_i^{(m)} I(y_i = -1)},
\qquad w_i^{(m)} = e^{-y_if_{m-1}(x_i)}.$$

**Rappel.** (10.30) : $\hat\gamma_{jm} = \arg\min_\gamma\sum_{x_i\in R_{jm}} L\big(y_i, f_{m-1}(x_i)+\gamma\big)$.

**Solution.** Avec $L(y,f) = e^{-yf}$ et $e^{a+b} = e^ae^b$ :

$$\sum_{x_i\in R_{jm}} e^{-y_i(f_{m-1}(x_i)+\gamma)} = \sum_{x_i\in R_{jm}} w_i^{(m)}e^{-y_i\gamma}
= e^{-\gamma}\,W_+ + e^{\gamma}\,W_-,$$

où $W_+ = \sum_{x_i\in R_{jm}} w_i^{(m)}I(y_i=1)$ et $W_- = \sum_{x_i\in R_{jm}} w_i^{(m)}I(y_i=-1)$ (on a
séparé les $y_i = 1$, pour lesquels $e^{-y_i\gamma} = e^{-\gamma}$, et les $y_i=-1$).

La fonction $\psi(\gamma) = W_+e^{-\gamma} + W_-e^{\gamma}$ est strictement convexe
($\psi'' = \psi > 0$) ; si $W_+>0$ et $W_->0$, elle tend vers $+\infty$ aux deux bouts et son
minimum est le zéro de $\psi'(\gamma) = -W_+e^{-\gamma} + W_-e^{\gamma}$ :

$$e^{2\gamma} = \frac{W_+}{W_-} \iff \hat\gamma_{jm} = \frac12\log\frac{W_+}{W_-}.$$

C'est (10.32) : la moitié du log-rapport de cotes **pondéré** dans la feuille, version
empirique de (10.16). Si la feuille est pure ($W_- = 0$ ou $W_+ = 0$), la valeur optimale
est infinie : en pratique on régularise (par exemple en ajoutant une petite constante aux deux sommes).

In [2]:
rng = np.random.RandomState(0)
yy = np.where(rng.rand(40) < 0.7, 1, -1)
ww = rng.exponential(size=40)
psi = lambda g: np.sum(ww * np.exp(-yy * g))
g_num = optimize.minimize_scalar(psi).x
g_formula = 0.5 * np.log(ww[yy == 1].sum() / ww[yy == -1].sum())
print("gamma numérique = %.6f ; formule (10.32) = %.6f" % (g_num, g_formula))

gamma numérique = 0.865083 ; formule (10.32) = 0.865083


**Illustration : un AdaBoost « à valeurs réelles ».** Utilisons (10.32) dans un boosting
d'arbres : à chaque étape, on prend les régions d'un arbre de moindres carrés pondérés
(avec les poids $w_i^{(m)}$) puis les valeurs (10.32) dans les feuilles, au lieu de $\pm\beta_m$.
C'est une variante proche du « Real AdaBoost » de Friedman et al. (2000).

In [3]:
def exp_tree_boost(X, y, M, J, Xt, yt, eps=1e-3):
    order = RegressionTree.presort(X)
    f, ft = np.zeros(len(y)), np.zeros(len(yt))
    errs = []
    for m in range(M):
        w = np.exp(-y * f)
        t = RegressionTree(J).fit(X, y, w, order=order)
        leaf = t.apply(X)
        for j in t.leaves:
            inl = leaf == j
            Wp = w[inl & (y == 1)].sum() + eps * w.mean()
            Wm = w[inl & (y == -1)].sum() + eps * w.mean()
            t.value[j] = 0.5 * np.log(Wp / Wm)        # (10.32), régularisé
        f += t.value[leaf]
        ft += t.predict(Xt)
        errs.append(np.mean(np.where(ft >= 0, 1, -1) != yt))
    return np.array(errs)

X, y = simulate_nested_spheres(2000, rng=np.random.RandomState(0))
Xt, yt = simulate_nested_spheres(5000, rng=np.random.RandomState(1))
err_real = exp_tree_boost(X, y, 200, 2, Xt, yt)
err_ada = AdaBoostM1(200, 2).fit(X, y).staged_errors(Xt, yt)

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(np.arange(1, 201), err_ada, color=BLUE, label="AdaBoost.M1 (feuilles $\\pm\\beta_m$)")
plt.plot(np.arange(1, 201), err_real, color=ORANGE, label="feuilles (10.32)")
plt.title("Souches boostées avec la perte exponentielle")
plt.xlabel("itérations")
plt.ylabel("erreur de test")
plt.legend()
plt.show()

Laisser chaque feuille choisir sa propre valeur (au lieu d'un même $\pm\beta_m$ pour tout
l'arbre) accélère nettement la décroissance de l'erreur.

> **À retenir.** Pour la perte exponentielle et des régions fixées, la valeur optimale d'une
> feuille est la moitié du log-rapport de cotes pondéré $\frac12\log(W_+/W_-)$.

## Exercice 10.8 : Pas de Newton pour la déviance multinomiale

**Énoncé (reformulé).** Problème à $K$ classes, $y_{ik} = 1$ si l'observation $i$ est dans la
classe $k$, 0 sinon. Le modèle courant $f_k(x)$, $k=1,\dots,K$, vérifie $\sum_k f_k(x) = 0$ (voir
(10.21)). On veut le mettre à jour dans une région $R$ en ajoutant des constantes :
$f_k(x) + \gamma_k$, avec $\gamma_K = 0$.
(a) Écrire la log-vraisemblance multinomiale et ses dérivées premières et secondes.
(b) En n'utilisant que la diagonale du hessien et en partant de $\gamma = 0$, montrer qu'un
pas de Newton approché donne
$\gamma_k^1 = \dfrac{\sum_{x_i\in R}(y_{ik}-p_{ik})}{\sum_{x_i\in R}p_{ik}(1-p_{ik})}$, $k = 1,\dots,K-1$ (éq. (10.57)),
avec $p_{ik} = e^{f_k(x_i)}/\sum_\ell e^{f_\ell(x_i)}$.
(c) On préfère une mise à jour de somme nulle. Par un argument de symétrie, montrer que
$\hat\gamma_k = \frac{K-1}{K}\big(\gamma_k^1 - \frac1K\sum_{\ell=1}^K\gamma_\ell^1\big)$ (éq. (10.58)) convient, où $\gamma_k^1$
est défini par (10.57) pour **tous** les $k = 1,\dots,K$.

### (a) Log-vraisemblance et dérivées

Pour les observations de $R$, après mise à jour, $p_{ik}(\gamma) = \dfrac{e^{f_k(x_i)+\gamma_k}}{\sum_\ell e^{f_\ell(x_i)+\gamma_\ell}}$ et

$$\ell(\gamma) = \sum_{x_i\in R}\sum_{k=1}^K y_{ik}\log p_{ik}(\gamma)
= \sum_{x_i\in R}\Big[\sum_k y_{ik}(f_k(x_i)+\gamma_k) - \log\sum_\ell e^{f_\ell(x_i)+\gamma_\ell}\Big],$$

car $\sum_k y_{ik} = 1$. **Dérivée de la log-somme-exponentielle :**
$\frac{\partial}{\partial\gamma_k}\log\sum_\ell e^{f_\ell+\gamma_\ell} = \frac{e^{f_k+\gamma_k}}{\sum_\ell e^{f_\ell+\gamma_\ell}} = p_{ik}$. D'où

$$\frac{\partial\ell}{\partial\gamma_k} = \sum_{x_i\in R}\big(y_{ik} - p_{ik}\big),\qquad
\frac{\partial^2\ell}{\partial\gamma_k\partial\gamma_{k'}} = -\sum_{x_i\in R} p_{ik}\big(\delta_{kk'} - p_{ik'}\big),$$

en utilisant $\partial p_{ik}/\partial\gamma_{k'} = p_{ik}(\delta_{kk'} - p_{ik'})$ ($\delta_{kk'}=1$ si $k=k'$, 0 sinon,
dérivée d'un quotient). En particulier, la diagonale vaut $-\sum_{x_i\in R}p_{ik}(1-p_{ik})$.

### (b) Pas de Newton diagonal

La méthode de Newton pour maximiser $\ell$ fait $\gamma \leftarrow \gamma - H^{-1}\nabla\ell$. En
remplaçant $H$ par sa diagonale (de termes $-\sum p_{ik}(1-p_{ik}) < 0$), les coordonnées se
découplent et, en partant de $\gamma = 0$ (donc avec les $p_{ik}$ du modèle courant) :

$$\gamma_k^1 = 0 - \frac{\sum_{x_i\in R}(y_{ik}-p_{ik})}{-\sum_{x_i\in R}p_{ik}(1-p_{ik})}
= \frac{\sum_{x_i\in R}(y_{ik}-p_{ik})}{\sum_{x_i\in R}p_{ik}(1-p_{ik})},\qquad k=1,\dots,K-1,$$

c'est (10.57) ($\gamma_K = 0$ est fixé).

### (c) Symétrisation

Le choix $\gamma_K = 0$ est arbitraire : la classe $K$ joue un rôle particulier. Remarquons
que la vraisemblance est **invariante** par ajout d'une même constante à tous les $\gamma_k$
($p_{ik}$ ne change pas). Donc la mise à jour « avec la classe $K$ comme référence » est
équivalente à $\gamma_k^{(K)} - c$ pour toute constante $c$. Si l'on refait le calcul en prenant
chaque classe $r$ comme référence, on obtient les mêmes formules (10.57), mais les
directions diffèrent : dans le pas diagonal, la classe de référence reçoit 0 au lieu de $\gamma_r^1$.

Un choix **symétrique** (ne privilégiant aucune classe) consiste à calculer $\gamma_k^1$ par (10.57)
pour **toutes** les classes, puis à projeter sur l'hyperplan $\sum_k\gamma_k = 0$ en retirant la
moyenne : $\gamma_k^1 - \frac1K\sum_\ell\gamma_\ell^1$. Le facteur $\frac{K-1}{K}$ vient de ce que, parmi les $K$
paramétrisations (une par classe de référence), chaque coordonnée $k$ n'est réellement
mise à jour que dans $K-1$ d'entre elles (elle vaut 0 quand $k$ est la référence) :
en moyennant les $K$ mises à jour (centrées), on obtient

$$\hat\gamma_k = \frac{K-1}{K}\Big(\gamma^1_k - \frac1K\sum_{\ell=1}^K\gamma^1_\ell\Big).$$

Par construction $\sum_k\hat\gamma_k = 0$ : le modèle mis à jour reste de somme nulle.